## 📚 **1. IMPORTACIÓN Y CONFIGURACIÓN**

In [ ]:
# Librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
from datetime import datetime

# Librerías para ARIMA y validación
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error
import optuna

# Configuraciones
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Librerías importadas correctamente")
print(f"📅 Fecha de análisis: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 📂 **2. CARGA DE DATOS**

In [ ]:
# Cargar datos del análisis exploratorio
try:
    with open('datasets_preparados.pkl', 'rb') as f:
        datasets = pickle.load(f)
    
    # Extraer datasets
    data = datasets['data']
    train_to1 = datasets['train_to1']
    train_to2 = datasets['train_to2']
    test_t1 = datasets['test_t1']
    test_to2 = datasets['test_to2']
    train_len = datasets['train_len']
    
    print("✅ Datos cargados exitosamente")
    print(f"📊 Configuración: {train_len} train / {len(data) - train_len} test")
    
except FileNotFoundError:
    print("❌ Error: No se encontró 'datasets_preparados.pkl'")
    print("💡 Ejecuta primero el notebook '01_Analisis_Exploratorio.ipynb'")

## 🔧 **3. FUNCIÓN ARIMA MEJORADA**

In [ ]:
# ==================== FUNCIÓN ARIMA CON VALIDACIONES ROBUSTAS ====================

def evaluar_expanding_forecast_arima(serie, order, window=12, step_size=1, horizon=1, metric='rmse'):
    """
    Simula pronósticos ARIMA en tiempo real con ventana expandida desde el inicio.
    Incluye validaciones de supuestos ARIMA mejoradas.
    
    Parameters:
    -----------
    serie : pd.Series
        Serie de tiempo a modelar
    order : tuple
        Orden ARIMA (p, d, q)
    window : int
        Número de puntos de validación
    step_size : int
        Tamaño del paso para validación
    horizon : int
        Horizonte de pronóstico
    metric : str
        Métrica de evaluación ('rmse' o 'mae')
    
    Returns:
    --------
    float
        Valor de la métrica de error
    """
    serie = pd.Series(serie).astype(float).dropna()
    n = len(serie)
    predichos = []
    observados = []
    p, d, q = order
    
    # Validación básica de parámetros
    if p < 0 or d < 0 or q < 0 or p > 5 or d > 2 or q > 5:
        return np.inf
    
    puntos_finales = list(range(n - window * step_size, n - horizon + 1, step_size))
    
    if len(puntos_finales) == 0:
        return np.inf
    
    for end_train in puntos_finales:
        train = serie.iloc[:end_train]
        test = serie.iloc[end_train:end_train + horizon]
        
        # Validaciones básicas
        if len(test) < horizon or test.isna().any() or len(train) < max(10, p + d + q + 5):
            continue
        
        try:
            # Suprimir warnings durante la optimización
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                # Configuración del modelo ARIMA con mejor manejo
                model = ARIMA(
                    train,
                    order=order,
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                    concentrate_scale=True  # Mejora la estabilidad numérica
                )
                
                fitted_model = model.fit(
                    method_kwargs={"warn_convergence": False},
                    low_memory=True
                )
                
                # Verificar convergencia del modelo
                if not fitted_model.mle_retvals['converged']:
                    continue
                    
                pred = fitted_model.forecast(steps=horizon)
                
                # Validar predicciones
                if not np.isfinite(pred).all() or np.any(np.abs(pred) > 1e6):
                    continue
                    
        except Exception as e:
            # Capturar errores específicos comunes
            if any(x in str(e) for x in ['singular', 'convergence', 'invert']):
                continue
            else:
                continue
        
        pred = pd.Series(pred).astype(float)
        mask = (~pred.isna()) & (~test.isna())
        
        if mask.sum() == 0:
            continue
        
        predichos.extend(pred[mask].tolist())
        observados.extend(test[mask].tolist())
    
    if len(predichos) == 0:
        return np.inf
    
    predichos = np.array(predichos, dtype=float)
    observados = np.array(observados, dtype=float)
    
    mask_finite = np.isfinite(predichos) & np.isfinite(observados)
    
    if mask_finite.sum() == 0:
        return np.inf
    
    predichos = predichos[mask_finite]
    observados = observados[mask_finite]
    
    # Penalizar si muy pocas predicciones válidas
    if len(predichos) < max(3, window // 3):
        return np.inf
    
    if metric == 'rmse':
        return np.sqrt(mean_squared_error(observados, predichos))
    elif metric == 'mae':
        return np.mean(np.abs(observados - predichos))
    else:
        raise ValueError("Metric must be 'rmse' or 'mae'")

print("✅ Función ARIMA mejorada definida")

## 🎯 **4. OPTIMIZACIÓN ARIMA - PRODUCTO 1**

In [ ]:
# Función objetivo para optimización ARIMA - Producto 1
def objective_arima_p1(trial):
    """Función objetivo mejorada para optimización de ARIMA - Producto 1"""
    p = trial.suggest_int("p", 0, 4)  # Reducir rango para estabilidad
    d = trial.suggest_int("d", 0, 2)
    q = trial.suggest_int("q", 0, 4)  # Reducir rango para estabilidad
    
    # Evitar combinaciones problemáticas
    if p + q > 6:  # Evitar modelos muy complejos
        return np.inf
    
    if p == 0 and q == 0 and d == 0:  # Modelo trivial
        return np.inf
        
    order = (p, d, q)
    return evaluar_expanding_forecast_arima(
        train_to1["producto1"],
        order,
        window=8,  # Aumentar ventana para más estabilidad
        step_size=1,
        horizon=1,
        metric='rmse'
    )

print("📈 OPTIMIZACIÓN ARIMA - PRODUCTO 1")
print("=" * 50)
print("🚀 Iniciando optimización con Optuna...")
print("⏱️ Con validaciones mejoradas y supuestos verificados")
print("-" * 60)

study_arima_p1 = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()  # Pruning para eficiencia
)
study_arima_p1.optimize(objective_arima_p1, n_trials=100, timeout=600)  # Más trials con timeout

best_order_p1 = (
    study_arima_p1.best_params['p'],
    study_arima_p1.best_params['d'],
    study_arima_p1.best_params['q']
)

print("\n" + "=" * 60)
print("RESULTADOS OPTIMIZACIÓN ARIMA - PRODUCTO 1")
print("=" * 60)
print(f"Mejor RMSE: {study_arima_p1.best_value:.4f}")
print(f"Mejor orden ARIMA: {best_order_p1}")
print(f"Número de trials completados: {len(study_arima_p1.trials)}")
print(f"Número de trials exitosos: {len([t for t in study_arima_p1.trials if t.state.name == 'COMPLETE' and t.value < np.inf])}")
print("=" * 60)

## 📊 **5. VISUALIZACIÓN Y ANÁLISIS DE SUPUESTOS - PRODUCTO 1**

In [ ]:
# Visualización de Convergencia ARIMA Producto 1
import matplotlib.pyplot as plt
import numpy as np

# Extraer datos de los trials completados
trials_df = pd.DataFrame([
    {**trial.params, 'rmse': trial.value}
    for trial in study_arima_p1.trials
    if trial.state.name == 'COMPLETE' and trial.value < np.inf
])

if len(trials_df) > 0:
    # Crear gráfico
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Configurar coordenadas
    params = ['p', 'd', 'q']
    x_pos = np.arange(len(params))
    rmse_values = trials_df['rmse'].values
    colors = plt.cm.viridis(1 - (rmse_values - rmse_values.min()) / (rmse_values.max() - rmse_values.min()))
    
    # Dibujar líneas
    for i, row in trials_df.iterrows():
        y_values = [row[param] for param in params]
        ax.plot(x_pos, y_values, color=colors[i], alpha=0.6, linewidth=0.8)
    
    # Configuración
    ax.set_xticks(x_pos)
    ax.set_xticklabels(params)
    ax.set_ylabel('Valor del Parámetro')
    ax.set_title('📈 Convergencia ARIMA Producto 1 (Amarillo=Mejor, Púrpura=Peor)')
    ax.grid(True, alpha=0.3)
    
    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=rmse_values.min(), vmax=rmse_values.max()))
    plt.colorbar(sm, ax=ax, label='RMSE')
    
    plt.tight_layout()
    plt.show()
    
    # Análisis de supuestos del mejor modelo ARIMA
    print(f"\n{'='*60}")
    print("📊 ANÁLISIS DE SUPUESTOS DEL MEJOR MODELO ARIMA - PRODUCTO 1")
    print(f"{'='*60}")
    print(f"🎯 Modelo: ARIMA{best_order_p1}")
    print(f"📈 RMSE: {study_arima_p1.best_value:.4f}")
    print(f"📊 Trials válidos: {len(trials_df)} de {len(study_arima_p1.trials)}")
    
    # Entrenar modelo final para análisis
    try:
        final_model = ARIMA(train_to1["producto1"], order=best_order_p1).fit()
        
        print(f"\n✅ VALIDACIONES DEL MODELO:")
        print(f"   • Modelo convergió: {final_model.mle_retvals['converged']}")
        print(f"   • Log-likelihood: {final_model.llf:.2f}")
        print(f"   • AIC: {final_model.aic:.2f}")
        print(f"   • BIC: {final_model.bic:.2f}")
        
        # Verificar residuos
        residuos = final_model.resid
        ljung_box = acorr_ljungbox(residuos, lags=10, return_df=True)
        p_value_ljung = ljung_box['lb_pvalue'].iloc[-1]
        
        print(f"\n🔍 SUPUESTOS ESTADÍSTICOS:")
        print(f"   • Test Ljung-Box (p-value): {p_value_ljung:.4f}")
        if p_value_ljung > 0.05:
            print("     → Residuos parecen ruido blanco (✅)")
        else:
            print("     → Posible autocorrelación en residuos (⚠️)")
            
        # Verificar normalidad de residuos
        _, p_norm = stats.jarque_bera(residuos)
        print(f"   • Test Jarque-Bera normalidad (p-value): {p_norm:.4f}")
        if p_norm > 0.05:
            print("     → Residuos parecen normales (✅)")
        else:
            print("     → Residuos no son normales (⚠️)")
            
        # Guardar modelo para usar después
        modelo_final_p1 = final_model
            
    except Exception as e:
        print(f"⚠️ Error en análisis: {str(e)}")
        modelo_final_p1 = None
        
else:
    print("⚠️ No se encontraron trials válidos para visualización")
    print("Esto puede indicar que los parámetros ARIMA necesitan ajuste")
    modelo_final_p1 = None

## 🎯 **6. OPTIMIZACIÓN ARIMA - PRODUCTO 2**

In [ ]:
# Función objetivo para optimización ARIMA - Producto 2
def objective_arima_p2(trial):
    """Función objetivo mejorada para optimización de ARIMA - Producto 2"""
    p = trial.suggest_int("p", 0, 4)
    d = trial.suggest_int("d", 0, 2)
    q = trial.suggest_int("q", 0, 4)
    
    # Evitar combinaciones problemáticas
    if p + q > 6:
        return np.inf
    
    if p == 0 and q == 0 and d == 0:
        return np.inf
        
    order = (p, d, q)
    return evaluar_expanding_forecast_arima(
        train_to2["producto2"],
        order,
        window=8,
        step_size=1,
        horizon=1,
        metric='rmse'
    )

print("📈 OPTIMIZACIÓN ARIMA - PRODUCTO 2")
print("=" * 50)
print("🚀 Iniciando optimización con Optuna...")
print("-" * 60)

study_arima_p2 = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)
study_arima_p2.optimize(objective_arima_p2, n_trials=100, timeout=600)

best_order_p2 = (
    study_arima_p2.best_params['p'],
    study_arima_p2.best_params['d'],
    study_arima_p2.best_params['q']
)

print("\n" + "=" * 60)
print("RESULTADOS OPTIMIZACIÓN ARIMA - PRODUCTO 2")
print("=" * 60)
print(f"Mejor RMSE: {study_arima_p2.best_value:.4f}")
print(f"Mejor orden ARIMA: {best_order_p2}")
print(f"Número de trials completados: {len(study_arima_p2.trials)}")
print(f"Número de trials exitosos: {len([t for t in study_arima_p2.trials if t.state.name == 'COMPLETE' and t.value < np.inf])}")
print("=" * 60)

## 📊 **7. ANÁLISIS DE SUPUESTOS - PRODUCTO 2**

In [ ]:
# Análisis similar para Producto 2
trials_df_p2 = pd.DataFrame([
    {**trial.params, 'rmse': trial.value}
    for trial in study_arima_p2.trials
    if trial.state.name == 'COMPLETE' and trial.value < np.inf
])

if len(trials_df_p2) > 0:
    # Visualización convergencia Producto 2
    fig, ax = plt.subplots(figsize=(10, 6))
    
    params = ['p', 'd', 'q']
    x_pos = np.arange(len(params))
    rmse_values_p2 = trials_df_p2['rmse'].values
    colors_p2 = plt.cm.plasma(1 - (rmse_values_p2 - rmse_values_p2.min()) / (rmse_values_p2.max() - rmse_values_p2.min()))
    
    for i, row in trials_df_p2.iterrows():
        y_values = [row[param] for param in params]
        ax.plot(x_pos, y_values, color=colors_p2[i], alpha=0.6, linewidth=0.8)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(params)
    ax.set_ylabel('Valor del Parámetro')
    ax.set_title('📈 Convergencia ARIMA Producto 2 (Amarillo=Mejor, Púrpura=Peor)')
    ax.grid(True, alpha=0.3)
    
    sm = plt.cm.ScalarMappable(cmap=plt.cm.plasma, norm=plt.Normalize(vmin=rmse_values_p2.min(), vmax=rmse_values_p2.max()))
    plt.colorbar(sm, ax=ax, label='RMSE')
    
    plt.tight_layout()
    plt.show()
    
    # Análisis de supuestos Producto 2
    print(f"\n{'='*60}")
    print("📊 ANÁLISIS DE SUPUESTOS DEL MEJOR MODELO ARIMA - PRODUCTO 2")
    print(f"{'='*60}")
    print(f"🎯 Modelo: ARIMA{best_order_p2}")
    print(f"📈 RMSE: {study_arima_p2.best_value:.4f}")
    print(f"📊 Trials válidos: {len(trials_df_p2)} de {len(study_arima_p2.trials)}")
    
    try:
        final_model_p2 = ARIMA(train_to2["producto2"], order=best_order_p2).fit()
        
        print(f"\n✅ VALIDACIONES DEL MODELO:")
        print(f"   • Modelo convergió: {final_model_p2.mle_retvals['converged']}")
        print(f"   • Log-likelihood: {final_model_p2.llf:.2f}")
        print(f"   • AIC: {final_model_p2.aic:.2f}")
        print(f"   • BIC: {final_model_p2.bic:.2f}")
        
        residuos_p2 = final_model_p2.resid
        ljung_box_p2 = acorr_ljungbox(residuos_p2, lags=10, return_df=True)
        p_value_ljung_p2 = ljung_box_p2['lb_pvalue'].iloc[-1]
        
        print(f"\n🔍 SUPUESTOS ESTADÍSTICOS:")
        print(f"   • Test Ljung-Box (p-value): {p_value_ljung_p2:.4f}")
        if p_value_ljung_p2 > 0.05:
            print("     → Residuos parecen ruido blanco (✅)")
        else:
            print("     → Posible autocorrelación en residuos (⚠️)")
            
        _, p_norm_p2 = stats.jarque_bera(residuos_p2)
        print(f"   • Test Jarque-Bera normalidad (p-value): {p_norm_p2:.4f}")
        if p_norm_p2 > 0.05:
            print("     → Residuos parecen normales (✅)")
        else:
            print("     → Residuos no son normales (⚠️)")
            
        modelo_final_p2 = final_model_p2
            
    except Exception as e:
        print(f"⚠️ Error en análisis: {str(e)}")
        modelo_final_p2 = None
        
else:
    print("⚠️ No se encontraron trials válidos para visualización Producto 2")
    modelo_final_p2 = None

## 🔮 **8. PRONÓSTICOS ARIMA**

In [ ]:
# Generar pronósticos con intervalos de confianza
horizonte_pronostico = 12

print(f"🔮 GENERANDO PRONÓSTICOS ARIMA ({horizonte_pronostico} períodos)")
print("=" * 60)

# Datos completos para entrenamiento final
datos_completos_p1 = pd.concat([train_to1['producto1'], test_t1['producto1']])
datos_completos_p2 = pd.concat([train_to2['producto2'], test_to2['producto2']])

# Pronósticos Producto 1
if modelo_final_p1 is not None:
    try:
        # Entrenar en datos completos
        modelo_completo_p1 = ARIMA(datos_completos_p1, order=best_order_p1).fit()
        
        # Generar pronósticos con intervalos
        forecast_p1 = modelo_completo_p1.get_forecast(steps=horizonte_pronostico)
        pronostico_p1 = forecast_p1.predicted_mean
        conf_int_p1 = forecast_p1.conf_int()
        
        print(f"✅ Producto 1 ARIMA{best_order_p1}:")
        print(f"   • Último pronóstico: {pronostico_p1.iloc[-1]:.2f}")
        print(f"   • Rango pronósticos: [{pronostico_p1.min():.2f}, {pronostico_p1.max():.2f}]")
        
    except Exception as e:
        print(f"⚠️ Error en pronósticos Producto 1: {str(e)}")
        pronostico_p1 = None
        conf_int_p1 = None
else:
    print("⚠️ No se puede generar pronósticos para Producto 1")
    pronostico_p1 = None
    conf_int_p1 = None

# Pronósticos Producto 2
if modelo_final_p2 is not None:
    try:
        modelo_completo_p2 = ARIMA(datos_completos_p2, order=best_order_p2).fit()
        
        forecast_p2 = modelo_completo_p2.get_forecast(steps=horizonte_pronostico)
        pronostico_p2 = forecast_p2.predicted_mean
        conf_int_p2 = forecast_p2.conf_int()
        
        print(f"✅ Producto 2 ARIMA{best_order_p2}:")
        print(f"   • Último pronóstico: {pronostico_p2.iloc[-1]:.2f}")
        print(f"   • Rango pronósticos: [{pronostico_p2.min():.2f}, {pronostico_p2.max():.2f}]")
        
    except Exception as e:
        print(f"⚠️ Error en pronósticos Producto 2: {str(e)}")
        pronostico_p2 = None
        conf_int_p2 = None
else:
    print("⚠️ No se puede generar pronósticos para Producto 2")
    pronostico_p2 = None
    conf_int_p2 = None

In [ ]:
# Visualizar pronósticos con intervalos de confianza
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))

# Producto 1
ax1.plot(datos_completos_p1.index, datos_completos_p1, label='Datos Históricos', 
         color='steelblue', linewidth=2, alpha=0.8)

if pronostico_p1 is not None:
    future_index_p1 = range(len(datos_completos_p1), len(datos_completos_p1) + horizonte_pronostico)
    
    ax1.plot(future_index_p1, pronostico_p1, label=f'ARIMA{best_order_p1}', 
             color='red', linewidth=2.5, marker='o', markersize=5)
    
    # Intervalos de confianza
    if conf_int_p1 is not None:
        ax1.fill_between(future_index_p1, 
                        conf_int_p1.iloc[:, 0], 
                        conf_int_p1.iloc[:, 1],
                        color='red', alpha=0.2, label='IC 95%')

ax1.axvline(x=len(datos_completos_p1)-1, color='black', linestyle='--', alpha=0.7, linewidth=2)
ax1.set_title(f'🔮 Pronósticos ARIMA - Producto 1 | Modelo: {best_order_p1}', 
              fontsize=14, fontweight='bold')
ax1.set_ylabel('Valores')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Producto 2
ax2.plot(datos_completos_p2.index, datos_completos_p2, label='Datos Históricos', 
         color='steelblue', linewidth=2, alpha=0.8)

if pronostico_p2 is not None:
    future_index_p2 = range(len(datos_completos_p2), len(datos_completos_p2) + horizonte_pronostico)
    
    ax2.plot(future_index_p2, pronostico_p2, label=f'ARIMA{best_order_p2}', 
             color='red', linewidth=2.5, marker='o', markersize=5)
    
    if conf_int_p2 is not None:
        ax2.fill_between(future_index_p2, 
                        conf_int_p2.iloc[:, 0], 
                        conf_int_p2.iloc[:, 1],
                        color='red', alpha=0.2, label='IC 95%')

ax2.axvline(x=len(datos_completos_p2)-1, color='black', linestyle='--', alpha=0.7, linewidth=2)
ax2.set_title(f'🔮 Pronósticos ARIMA - Producto 2 | Modelo: {best_order_p2}', 
              fontsize=14, fontweight='bold')
ax2.set_ylabel('Valores')
ax2.set_xlabel('Tiempo')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📊 **9. RESUMEN Y GUARDADO**

In [ ]:
# Resumen comparativo
print("📊 RESUMEN MODELOS ARIMA")
print("=" * 60)

print(f"📦 PRODUCTO 1:")
print(f"   • Mejor modelo: ARIMA{best_order_p1}")
print(f"   • RMSE: {study_arima_p1.best_value:.4f}")
print(f"   • Trials exitosos: {len([t for t in study_arima_p1.trials if t.state.name == 'COMPLETE' and t.value < np.inf])}/{len(study_arima_p1.trials)}")
if pronostico_p1 is not None:
    print(f"   • Pronóstico final: {pronostico_p1.iloc[-1]:.2f}")

print(f"\n📦 PRODUCTO 2:")
print(f"   • Mejor modelo: ARIMA{best_order_p2}")
print(f"   • RMSE: {study_arima_p2.best_value:.4f}")
print(f"   • Trials exitosos: {len([t for t in study_arima_p2.trials if t.state.name == 'COMPLETE' and t.value < np.inf])}/{len(study_arima_p2.trials)}")
if pronostico_p2 is not None:
    print(f"   • Pronóstico final: {pronostico_p2.iloc[-1]:.2f}")

# Guardar resultados
resultados_arima = {
    'p1': {
        'study': study_arima_p1,
        'best_order': best_order_p1,
        'best_rmse': study_arima_p1.best_value,
        'modelo': modelo_final_p1 if 'modelo_final_p1' in locals() else None,
        'pronostico': pronostico_p1 if 'pronostico_p1' in locals() else None,
        'conf_int': conf_int_p1 if 'conf_int_p1' in locals() else None
    },
    'p2': {
        'study': study_arima_p2,
        'best_order': best_order_p2,
        'best_rmse': study_arima_p2.best_value,
        'modelo': modelo_final_p2 if 'modelo_final_p2' in locals() else None,
        'pronostico': pronostico_p2 if 'pronostico_p2' in locals() else None,
        'conf_int': conf_int_p2 if 'conf_int_p2' in locals() else None
    }
}

with open('resultados_arima.pkl', 'wb') as f:
    pickle.dump(resultados_arima, f)

print("\n💾 Resultados guardados en 'resultados_arima.pkl'")
print("\n" + "=" * 60)
print("🚀 ANÁLISIS ARIMA COMPLETADO")
print("📝 Continúa con el notebook '04_Prophet.ipynb'")
print("=" * 60)